In [ ]:

!pip install pyspark delta-spark

import pyspark
from delta import *

builder = pyspark.sql.SparkSession.builder.appName("DeltaAssignment") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()


In [ ]:
# Loading dataset
path = "/content/customer_master.csv"

master=spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(path)

print("Raw Master Data:")
master.show()

Raw Master Data:
+-----------+-------------+-----------------+--------+--------+
|customer_id|         name|            email|   phone|location|
+-----------+-------------+-----------------+--------+--------+
|        101|  Alice Smith|  alice@email.com|555-0101|New York|
|        102|    Bob Jones|    bob@email.com|555-0202| Chicago|
|        103|Charlie Brown|charlie@email.com|555-0303| Seattle|
|        103|Charlie Brown|charlie@email.com|555-0303| Seattle|
+-----------+-------------+-----------------+--------+--------+



In [ ]:
# . Cleaning the data
df_clean = master.dropDuplicates().dropna()

print("Cleaned Master Data:")
df_clean.show()

Cleaned Master Data:
+-----------+-------------+-----------------+--------+--------+
|customer_id|         name|            email|   phone|location|
+-----------+-------------+-----------------+--------+--------+
|        103|Charlie Brown|charlie@email.com|555-0303| Seattle|
|        102|    Bob Jones|    bob@email.com|555-0202| Chicago|
|        101|  Alice Smith|  alice@email.com|555-0101|New York|
+-----------+-------------+-----------------+--------+--------+



In [ ]:
# 3. Saveing the clean dataframe in Delta table format
delta_path = "/content/customer_delta"

df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .save(delta_path)

print("Master Delta table successfully created in Colab!")

Master Delta table successfully created in Colab!


In [ ]:
from delta.tables import DeltaTable

incremental_path = "/content/customer_incremental.csv"
incremental = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(incremental_path)

print(" new data :")
incremental.show()

# Connect to Delta table
target_table = DeltaTable.forPath(spark, "/content/customer_delta")

# Scd type 1
target_table.alias("target").merge(
    incremental.alias("source"),
    "target.customer_id = source.customer_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

#  merged result
print("updated Delta Table:")
spark.read.format("delta").load("/content/customer_delta").orderBy("customer_id").show()

 new data :
+-----------+-----------+---------------+--------+--------+
|customer_id|       name|          email|   phone|location|
+-----------+-----------+---------------+--------+--------+
|        101|Alice Smith|alice@email.com|555-0101|  Boston|
|        102|  Bob Jones|  bob@email.com|999-9999| Chicago|
|        104|  David Lee|david@email.com|555-0404|  Austin|
+-----------+-----------+---------------+--------+--------+

updated Delta Table:
+-----------+-------------+-----------------+--------+--------+
|customer_id|         name|            email|   phone|location|
+-----------+-------------+-----------------+--------+--------+
|        101|  Alice Smith|  alice@email.com|555-0101|  Boston|
|        102|    Bob Jones|    bob@email.com|999-9999| Chicago|
|        103|Charlie Brown|charlie@email.com|555-0303| Seattle|
|        104|    David Lee|  david@email.com|555-0404|  Austin|
+-----------+-------------+-----------------+--------+--------+



In [ ]:
from pyspark.sql.functions import current_date, lit

# Adding the Scd Type 2 columns to cleaned master data
master_scd2 = df_clean.withColumn("start_date", current_date()) \
                           .withColumn("end_date", lit(None).cast("date")) \
                           .withColumn("is_current", lit(True))

# 2. Saveing it as new Delta table path
master_scd2.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/content/customer_delta_scd2")

spark.read.format("delta").load("/content/customer_delta_scd2").show()

+-----------+-------------+-----------------+--------+--------+----------+--------+----------+
|customer_id|         name|            email|   phone|location|start_date|end_date|is_current|
+-----------+-------------+-----------------+--------+--------+----------+--------+----------+
|        103|Charlie Brown|charlie@email.com|555-0303| Seattle|2026-07-04|    NULL|      true|
|        102|    Bob Jones|    bob@email.com|555-0202| Chicago|2026-07-04|    NULL|      true|
|        101|  Alice Smith|  alice@email.com|555-0101|New York|2026-07-04|    NULL|      true|
+-----------+-------------+-----------------+--------+--------+----------+--------+----------+



In [ ]:
from delta.tables import DeltaTable

#. Connecting to scd type 2 dlta table
scd2_table = DeltaTable.forPath(spark, "/content/customer_delta_scd2")


scd2_table.alias("target").merge(
    incremental.alias("source"),
    "target.customer_id = source.customer_id AND target.is_current = true"
).whenMatchedUpdate(set = {
    "is_current": "false",
    "end_date": "current_date()"
}).execute()

# adding tracking columns t o new data and append them as active rows
new_active_records = incremental.withColumn("start_date", current_date()) \
                                 .withColumn("end_date", lit(None).cast("date")) \
                                 .withColumn("is_current", lit(True))

new_active_records.write \
    .format("delta") \
    .mode("append") \
    .save("/content/customer_delta_scd2")

print("Scd Type 2 complete! historical Log:")
spark.read.format("delta").load("/content/customer_delta_scd2").orderBy("customer_id", "start_date").show()

Scd Type 2 complete! historical Log:
+-----------+-------------+-----------------+--------+--------+----------+----------+----------+
|customer_id|         name|            email|   phone|location|start_date|  end_date|is_current|
+-----------+-------------+-----------------+--------+--------+----------+----------+----------+
|        101|  Alice Smith|  alice@email.com|555-0101|New York|2026-07-04|2026-07-04|     false|
|        101|  Alice Smith|  alice@email.com|555-0101|  Boston|2026-07-04|      NULL|      true|
|        102|    Bob Jones|    bob@email.com|555-0202| Chicago|2026-07-04|2026-07-04|     false|
|        102|    Bob Jones|    bob@email.com|999-9999| Chicago|2026-07-04|      NULL|      true|
|        103|Charlie Brown|charlie@email.com|555-0303| Seattle|2026-07-04|      NULL|      true|
|        104|    David Lee|  david@email.com|555-0404|  Austin|2026-07-04|      NULL|      true|
+-----------+-------------+-----------------+--------+--------+----------+----------+-----

In [ ]:
# Loading the final SCd Type 2 table
final_table = spark.read.format("delta").load("/content/customer_delta_scd2")

#  Validateing total row count (Historical + active)
total_rows = final_table.count()
print(f"Total rows in historical table: {total_rows}")

#  validate no duplicates in  active records
active_records = final_table.filter("is_current = true")
active_count = active_records.count()
distinct_active = active_records.select("customer_id").distinct().count()

print("-" * 40)
if active_count == distinct_active:
    print(f"Validation passed")
    print(f"{active_count} active customers with no duplicates.")
else:
    print("Duplicates  in active records")
print("-" * 40)

# final active dataset
print("Final active dataset :")
active_records.drop("start_date", "end_date", "is_current").orderBy("customer_id").show()

Total rows in historical table: 6
----------------------------------------
Validation passed
4 active customers with no duplicates.
----------------------------------------
Final active dataset :
+-----------+-------------+-----------------+--------+--------+
|customer_id|         name|            email|   phone|location|
+-----------+-------------+-----------------+--------+--------+
|        101|  Alice Smith|  alice@email.com|555-0101|  Boston|
|        102|    Bob Jones|    bob@email.com|999-9999| Chicago|
|        103|Charlie Brown|charlie@email.com|555-0303| Seattle|
|        104|    David Lee|  david@email.com|555-0404|  Austin|
+-----------+-------------+-----------------+--------+--------+

